# 📈 Time Series Forecasting using RNN, LSTM, and GRU
### Stock Price Prediction — Apple Inc. (AAPL)

**Author:** Haneen Nasser  
**Goal:** Compare three deep learning architectures (Simple RNN, LSTM, GRU) on the same stock price dataset and evaluate their forecasting accuracy.

---
**Models:**
- 🔵 Simple RNN
- 🟢 LSTM (Long Short-Term Memory)
- 🟠 GRU (Gated Recurrent Unit)

**Metrics:** RMSE, MAE, R² Score

## 1. Install & Import Libraries

In [ ]:
# Install yfinance to download stock data
!pip install yfinance --quiet

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import yfinance as yf

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, LSTM, GRU, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print("✅ Libraries imported successfully")
print(f"TensorFlow version: {tf.__version__}")

## 2. Download & Explore Stock Data

In [ ]:
# Download Apple stock data (5 years)
ticker = 'AAPL'
df = yf.download(ticker, start='2018-01-01', end='2023-12-31', progress=False)

print(f"📊 Dataset shape: {df.shape}")
print(f"📅 Date range: {df.index[0].date()} → {df.index[-1].date()}")
print("\nFirst 5 rows:")
df.head()

In [ ]:
# Check for missing values
print("Missing values:")
print(df.isnull().sum())
print(f"\nBasic statistics:")
df['Close'].describe()

In [ ]:
# Plot the closing price
plt.figure(figsize=(14, 5))
plt.plot(df.index, df['Close'], color='steelblue', linewidth=1.5)
plt.title(f'{ticker} Closing Price (2018–2023)', fontsize=16, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Price (USD)')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()
print("✅ Raw data plotted")

## 3. Data Preprocessing

In [ ]:
# Use only the 'Close' price
data = df[['Close']].values

# Normalize data to [0, 1] range
scaler = MinMaxScaler(feature_range=(0, 1))
data_scaled = scaler.fit_transform(data)

# Train/Test split: 80% train, 20% test
train_size = int(len(data_scaled) * 0.80)
train_data = data_scaled[:train_size]
test_data  = data_scaled[train_size:]

print(f"Total samples  : {len(data_scaled)}")
print(f"Training samples: {len(train_data)}")
print(f"Testing samples : {len(test_data)}")

In [ ]:
def create_sequences(dataset, time_steps=60):
    """
    Create input sequences and corresponding labels.
    time_steps: number of past days used to predict the next day
    """
    X, y = [], []
    for i in range(time_steps, len(dataset)):
        X.append(dataset[i - time_steps:i, 0])
        y.append(dataset[i, 0])
    return np.array(X), np.array(y)

TIME_STEPS = 60  # Use last 60 days to predict next day

X_train, y_train = create_sequences(train_data, TIME_STEPS)
X_test,  y_test  = create_sequences(test_data,  TIME_STEPS)

# Reshape for RNN input: (samples, time_steps, features)
X_train = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
X_test  = X_test.reshape(X_test.shape[0],  X_test.shape[1],  1)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape : {X_test.shape}")
print(f"y_train shape: {y_train.shape}")

## 4. Build the Three Models

All models share the **same architecture depth** so the comparison is fair.

In [ ]:
def build_rnn(input_shape):
    model = Sequential([
        SimpleRNN(64, return_sequences=True, input_shape=input_shape),
        Dropout(0.2),
        SimpleRNN(32, return_sequences=False),
        Dropout(0.2),
        Dense(16, activation='relu'),
        Dense(1)
    ], name='SimpleRNN')
    model.compile(optimizer='adam', loss='mse')
    return model


def build_lstm(input_shape):
    model = Sequential([
        LSTM(64, return_sequences=True, input_shape=input_shape),
        Dropout(0.2),
        LSTM(32, return_sequences=False),
        Dropout(0.2),
        Dense(16, activation='relu'),
        Dense(1)
    ], name='LSTM')
    model.compile(optimizer='adam', loss='mse')
    return model


def build_gru(input_shape):
    model = Sequential([
        GRU(64, return_sequences=True, input_shape=input_shape),
        Dropout(0.2),
        GRU(32, return_sequences=False),
        Dropout(0.2),
        Dense(16, activation='relu'),
        Dense(1)
    ], name='GRU')
    model.compile(optimizer='adam', loss='mse')
    return model


INPUT_SHAPE = (TIME_STEPS, 1)

rnn_model  = build_rnn(INPUT_SHAPE)
lstm_model = build_lstm(INPUT_SHAPE)
gru_model  = build_gru(INPUT_SHAPE)

print("✅ All three models built")
lstm_model.summary()

## 5. Train the Models

In [ ]:
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

EPOCHS     = 50
BATCH_SIZE = 32

# --- Train Simple RNN ---
print("🔵 Training Simple RNN...")
rnn_history = rnn_model.fit(
    X_train, y_train,
    epochs=EPOCHS, batch_size=BATCH_SIZE,
    validation_split=0.1,
    callbacks=[early_stop],
    verbose=0
)
print(f"   Stopped at epoch {len(rnn_history.history['loss'])}")

# --- Train LSTM ---
print("🟢 Training LSTM...")
lstm_history = lstm_model.fit(
    X_train, y_train,
    epochs=EPOCHS, batch_size=BATCH_SIZE,
    validation_split=0.1,
    callbacks=[early_stop],
    verbose=0
)
print(f"   Stopped at epoch {len(lstm_history.history['loss'])}")

# --- Train GRU ---
print("🟠 Training GRU...")
gru_history = gru_model.fit(
    X_train, y_train,
    epochs=EPOCHS, batch_size=BATCH_SIZE,
    validation_split=0.1,
    callbacks=[early_stop],
    verbose=0
)
print(f"   Stopped at epoch {len(gru_history.history['loss'])}")

print("\n✅ All models trained!")

In [ ]:
# Plot training loss curves
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
configs = [
    (rnn_history,  'Simple RNN', 'steelblue'),
    (lstm_history, 'LSTM',       'seagreen'),
    (gru_history,  'GRU',        'darkorange'),
]

for ax, (hist, name, color) in zip(axes, configs):
    ax.plot(hist.history['loss'],     label='Train Loss', color=color)
    ax.plot(hist.history['val_loss'], label='Val Loss',   color=color, linestyle='--', alpha=0.7)
    ax.set_title(f'{name} — Loss Curve', fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('MSE Loss')
    ax.legend()
    ax.grid(alpha=0.3)

plt.suptitle('Training vs Validation Loss', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 6. Evaluate & Compare Models

In [ ]:
def evaluate_model(model, X_test, y_test, scaler, name):
    # Predict
    preds_scaled = model.predict(X_test, verbose=0)
    
    # Inverse transform to original price scale
    preds  = scaler.inverse_transform(preds_scaled)
    actual = scaler.inverse_transform(y_test.reshape(-1, 1))
    
    # Calculate metrics
    rmse = np.sqrt(mean_squared_error(actual, preds))
    mae  = mean_absolute_error(actual, preds)
    r2   = r2_score(actual, preds)
    
    return {
        'Model':      name,
        'RMSE':       round(rmse, 4),
        'MAE':        round(mae, 4),
        'R² Score':   round(r2, 4),
        'predictions': preds,
        'actual':      actual
    }

results = [
    evaluate_model(rnn_model,  X_test, y_test, scaler, 'Simple RNN'),
    evaluate_model(lstm_model, X_test, y_test, scaler, 'LSTM'),
    evaluate_model(gru_model,  X_test, y_test, scaler, 'GRU'),
]

# Display comparison table
metrics_df = pd.DataFrame([{k: v for k, v in r.items() if k not in ['predictions', 'actual']}
                            for r in results])
metrics_df = metrics_df.set_index('Model')
print("=" * 50)
print("       MODEL COMPARISON RESULTS")
print("=" * 50)
print(metrics_df.to_string())
print("=" * 50)
print("\n🏆 Best RMSE :", metrics_df['RMSE'].idxmin())
print("🏆 Best MAE  :", metrics_df['MAE'].idxmin())
print("🏆 Best R²   :", metrics_df['R² Score'].idxmax())

In [ ]:
# Plot actual vs predicted for all 3 models
actual_prices = results[0]['actual']  # Same for all

fig, axes = plt.subplots(3, 1, figsize=(14, 12))
colors = ['steelblue', 'seagreen', 'darkorange']

for ax, result, color in zip(axes, results, colors):
    ax.plot(actual_prices,          label='Actual Price',    color='black',  linewidth=1.5)
    ax.plot(result['predictions'],  label=f'{result["Model"]} Predicted', color=color, linewidth=1.5, alpha=0.85)
    ax.set_title(f'{result["Model"]} — RMSE: {result["RMSE"]} | MAE: {result["MAE"]} | R²: {result["R² Score"]}',
                 fontweight='bold')
    ax.set_ylabel('Stock Price (USD)')
    ax.legend()
    ax.grid(alpha=0.3)

axes[-1].set_xlabel('Test Days')
plt.suptitle('AAPL Stock Price — Actual vs Predicted', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Bar chart comparison of metrics
fig, axes = plt.subplots(1, 3, figsize=(14, 5))

model_names = [r['Model'] for r in results]
bar_colors  = ['steelblue', 'seagreen', 'darkorange']

rmse_vals = [r['RMSE']     for r in results]
mae_vals  = [r['MAE']      for r in results]
r2_vals   = [r['R² Score'] for r in results]

for ax, vals, title, ylabel in zip(
    axes,
    [rmse_vals, mae_vals, r2_vals],
    ['RMSE (lower is better)', 'MAE (lower is better)', 'R² Score (higher is better)'],
    ['RMSE', 'MAE', 'R²']
):
    bars = ax.bar(model_names, vals, color=bar_colors, edgecolor='white', width=0.5)
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel(ylabel)
    ax.grid(axis='y', alpha=0.3)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                str(val), ha='center', va='bottom', fontweight='bold', fontsize=10)

plt.suptitle('Model Comparison — All Metrics', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Conclusion

| Model | Strengths | Weaknesses |
|-------|-----------|------------|
| **Simple RNN** | Fast to train, simple architecture | Suffers from vanishing gradient on long sequences |
| **LSTM** | Handles long-term dependencies well | More parameters, slower to train |
| **GRU** | Faster than LSTM, comparable accuracy | Slightly less expressive than LSTM |

### Key Takeaways
- All three models use the **same dataset** (AAPL Closing Price 2018–2023) and the **same sequence length** (60 days)
- **LSTM** and **GRU** generally outperform Simple RNN on financial time series due to their gating mechanisms
- **GRU** often achieves results close to LSTM with fewer parameters — making it a practical choice
- Metrics used: **RMSE** (penalizes large errors), **MAE** (average error in dollars), **R²** (explained variance)

---
*Project by Haneen Nasser — Cairo University, CS & AI Department, 2024*